# Approach 2 - Not chosen (VLM)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1 · Settings

In [ ]:
DATASET_SRC = "/content/drive/MyDrive/archive.zip" 
OUT         = "/content/drive/MyDrive/motor-claim-runs"
DAMAGE_RUN = "final_r50_640_40ep"
PARTS_RUN  = "parts_frcnn_640"

SCORE_THR = 0.6 
SEED      = 42

import os
REPORT_DIR = f"{OUT}/evaluation"
VLM_DIR    = f"{OUT}/vlm"
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(VLM_DIR, exist_ok=True)

assert os.path.isdir(OUT), f"not found: {OUT} — is Drive mounted and is the path right?"
for _r in (DAMAGE_RUN, PARTS_RUN):
    assert os.path.isdir(f"{OUT}/{_r}"), (
        f"run folder missing: {OUT}/{_r}\n"
        f"available: {sorted(os.listdir(OUT))}")
print("runs found:", DAMAGE_RUN, "|", PARTS_RUN)

runs found: final_r50_640_40ep | parts_frcnn_640


## 2 · Install

In [ ]:
import sys
_already = "transformers" in sys.modules


def _too_old(mod):
    return tuple(int(x) for x in mod.__version__.split(".")[:2]) < (4, 46)


try:
    import transformers
    _need = _too_old(transformers)
except ImportError:
    transformers, _need = None, True

if _need:
    get_ipython().system('pip install -q -U "transformers>=4.46"')
    if _already and transformers is not None and _too_old(transformers):
        raise SystemExit(
            "transformers was upgraded but an old copy is already loaded in this "
            "kernel.\nRuntime > Restart session, then run from the top. Nothing "
            "is lost: every result is checkpointed on Drive."
        )
    import transformers

print("transformers", transformers.__version__)
assert not _too_old(transformers), "need transformers >= 4.46 for SmolVLM"

transformers 5.22.1


In [ ]:
import gc, json, random, re, shutil, time, unicodedata, zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from PIL import Image

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VLM_DTYPE = (torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
             else torch.float16 if DEVICE.type == "cuda" else torch.float32)
print("torch", torch.__version__, "| device", DEVICE, "| vlm dtype", VLM_DTYPE)
if DEVICE.type == "cuda":
    get_ipython().system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

torch 2.14.1+cu130 | device cuda | vlm dtype torch.bfloat16
NVIDIA L4, 23034 MiB


## 3 · Dataset

The two folders are named **the wrong way round** in the Kaggle download: the
"Car damages dataset" holds the 21 part classes and the "Car parts dataset"
holds the 8 damage classes. The roots are therefore identified by the class
names inside `meta.json`, never by folder name.

In [5]:
os.makedirs("/content/data", exist_ok=True)
if not os.listdir("/content/data"):
    if os.path.isdir(DATASET_SRC):
        shutil.copytree(DATASET_SRC, "/content/data/dataset", dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(DATASET_SRC) as z:
            z.extractall("/content/data")
else:
    print("/content/data already populated, skipping extract")

DAMAGE_HINTS = {"Scratch", "Dent", "Broken part", "Missing part", "Paint chip"}
PART_HINTS   = {"Hood", "Front-bumper", "Headlight", "Roof", "Windshield"}


def classes_in(root):
    meta = json.loads((Path(root) / "meta.json").read_text(encoding="utf-8"))
    return [c["title"] for c in meta["classes"]]


ROOTS = {}
for meta_path in sorted(Path("/content/data").rglob("meta.json")):
    root = meta_path.parent
    if not (root / "File1" / "ann").is_dir():
        continue
    names = set(classes_in(root))
    kind = "damage" if names & DAMAGE_HINTS else ("parts" if names & PART_HINTS else None)
    if kind and kind not in ROOTS:
        ROOTS[kind] = root

assert set(ROOTS) == {"damage", "parts"}, f"could not identify both datasets: {ROOTS}"
DAMAGE_ROOT, PARTS_ROOT = ROOTS["damage"], ROOTS["parts"]
for kind, root in ROOTS.items():
    print(f"{kind:7s} <- {root.name}  ({len(classes_in(root))} classes)")

parts   <- Car damages dataset  (21 classes)
damage  <- Car parts dataset  (8 classes)


## 4 · Ground truth for the held-out images

Only **label sets** are needed here, not boxes, so the full Supervisely parsing
and the `Dataset` class from the training notebook are not required.

1. the **class merge** — rare damage classes were folded into `Other damage`
2. the **degenerate-box filter** — polygons thinner than 1px were dropped

Both are read from the run's own `metrics.json` rather than re-declared, and the
test image list comes from its `splits.json`. The assertion at the end is the
guarantee that both approaches see the same images.

In [ ]:
MIN_BOX_SIDE = 1.0   
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def polygon_to_box(exterior, width, height):
    xs = [p[0] for p in exterior]
    ys = [p[1] for p in exterior]
    x1, x2 = max(0.0, min(xs)), min(float(width), max(xs))
    y1, y2 = max(0.0, min(ys)), min(float(height), max(ys))
    if x2 - x1 < MIN_BOX_SIDE or y2 - y1 < MIN_BOX_SIDE:
        return None
    return (x1, y1, x2, y2)


def labels_in_annotation(data, merge):
    """The class labels that survive to become training targets on this image."""
    width, height = int(data["size"]["width"]), int(data["size"]["height"])
    out = []
    for obj in data["objects"]:
        if obj.get("geometryType") != "polygon":
            continue
        ext = obj["points"]["exterior"]
        if len(ext) < 3 or polygon_to_box(ext, width, height) is None:
            continue
        out.append(merge.get(obj["classTitle"], obj["classTitle"]))
    return out


def test_truth(root, run_name, subset="File1"):
    """(classes, image paths, label sets) for the exact test split of a finished run."""
    meta = json.loads((Path(OUT) / run_name / "metrics.json").read_text())
    classes, merge = meta["classes"], meta["config"]["merge"] or {}
    wanted = set(json.loads((Path(OUT) / run_name / "splits.json").read_text())["test"])

    img_dir, ann_dir = Path(root) / subset / "img", Path(root) / subset / "ann"
    images = [p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
    by_name = {p.name: p for p in images}
    by_stem = {p.stem: p for p in images}

    paths, truth = [], []
    for ann_path in sorted(ann_dir.glob("*.json")):
        name = ann_path.name[: -len(".json")]
        image = by_name.get(name) or by_stem.get(Path(name).stem)
        if image is None or image.name not in wanted:
            continue
        labels = labels_in_annotation(
            json.loads(ann_path.read_text(encoding="utf-8")), merge)
        unknown = set(labels) - set(classes)
        assert not unknown, f"{ann_path.name}: classes not in metrics.json: {unknown}"
        paths.append(image)
        truth.append(set(labels))

    assert len(paths) == len(wanted), \
        f"{run_name}: matched {len(paths)} of {len(wanted)} test images"
    print(f"{run_name}: {len(paths)} held-out images, {len(classes)} classes")
    return classes, paths, truth


DAMAGE_CLASSES, damage_paths, damage_truth = test_truth(DAMAGE_ROOT, DAMAGE_RUN)
PARTS_CLASSES,  parts_paths,  parts_truth  = test_truth(PARTS_ROOT,  PARTS_RUN)

print("\ndamage classes:", DAMAGE_CLASSES)
print("parts classes  :", PARTS_CLASSES)
print(f"\nimages with no labelled object at all: "
      f"damage {sum(not t for t in damage_truth)}, parts {sum(not t for t in parts_truth)}")

final_r50_640_40ep: 81 held-out images, 6 classes
parts_frcnn_640: 100 held-out images, 21 classes

damage classes: ['Missing part', 'Broken part', 'Scratch', 'Other damage', 'Dent', 'Paint chip']
parts classes  : ['Quarter-panel', 'Front-wheel', 'Back-window', 'Trunk', 'Front-door', 'Rocker-panel', 'Grille', 'Windshield', 'Front-window', 'Back-door', 'Headlight', 'Back-wheel', 'Back-windshield', 'Hood', 'Fender', 'Tail-light', 'License-plate', 'Front-bumper', 'Back-bumper', 'Mirror', 'Roof']

images with no labelled object at all: damage 0, parts 0


## 5 · Approach 1 on the shared task

The trained detectors, with their boxes thrown away — only the class names are
kept. The backbone is inferred from the checkpoint's own weight names, so this
works for any of the runs without being told which architecture each used.

In [7]:
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_FPN_Weights, FasterRCNN_ResNet50_FPN_V2_Weights,
    MaskRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_mobilenet_v3_large_fpn,
    fasterrcnn_resnet50_fpn_v2, maskrcnn_resnet50_fpn_v2,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms import functional as TF


def build_faster_rcnn(num_classes, min_sizes, max_size):
    m = fasterrcnn_mobilenet_v3_large_fpn(
        weights=None, weights_backbone=None,
        min_size=tuple(min_sizes), max_size=max_size)
    m.roi_heads.box_predictor = FastRCNNPredictor(
        m.roi_heads.box_predictor.cls_score.in_features, num_classes + 1)
    return m


def build_faster_rcnn_r50(num_classes, min_sizes, max_size):
    m = fasterrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None,
        min_size=tuple(min_sizes), max_size=max_size)
    m.roi_heads.box_predictor = FastRCNNPredictor(
        m.roi_heads.box_predictor.cls_score.in_features, num_classes + 1)
    return m


def build_mask_rcnn(num_classes, min_sizes, max_size):
    m = maskrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None,
        min_size=tuple(min_sizes), max_size=max_size)
    m.roi_heads.box_predictor = FastRCNNPredictor(
        m.roi_heads.box_predictor.cls_score.in_features, num_classes + 1)
    m.roi_heads.mask_predictor = MaskRCNNPredictor(
        m.roi_heads.mask_predictor.conv5_mask.in_channels, 256, num_classes + 1)
    return m


def load_detector(run_name):
    """Rebuild the architecture recorded in best.pt and load its weights.

    The backbone is read off the state dict instead of being passed in: a
    ResNet-50 FPN has backbone.body.layer1.* keys, MobileNetV3 does not. Getting
    this wrong gives a load_state_dict error, never a wrong-but-plausible model.
    """
    ck = torch.load(Path(OUT) / run_name / "best.pt", map_location="cpu", weights_only=True)
    resnet = any(k.startswith("backbone.body.layer1.") for k in ck["model"])
    build = build_mask_rcnn if ck["with_masks"] else (
        build_faster_rcnn_r50 if resnet else build_faster_rcnn)
    model = build(len(ck["classes"]), ck["min_sizes"], ck["max_size"])
    model.load_state_dict(ck["model"])
    model.to(DEVICE).eval()
    print(f"{run_name}: {build.__name__}, {len(ck['classes'])} classes, "
          f"masks={ck['with_masks']}, best epoch {ck['epoch'] + 1}")
    return model


@torch.no_grad()
def detector_label_sets(run_name, paths, classes, score_thr=SCORE_THR):
    model = load_detector(run_name)
    sets = []
    try:
        for p in paths:
            with Image.open(p) as im:
                t = TF.to_tensor(im.convert("RGB")).to(DEVICE)
            o = model([t])[0]
            keep = o["scores"] >= score_thr
            sets.append({classes[int(l) - 1] for l in o["labels"][keep].tolist()})
    finally:
        del model
        gc.collect(); torch.cuda.empty_cache()
    return sets


damage_det = detector_label_sets(DAMAGE_RUN, damage_paths, DAMAGE_CLASSES)
parts_det  = detector_label_sets(PARTS_RUN,  parts_paths,  PARTS_CLASSES)
print(f"\ndetector label sets ready at score >= {SCORE_THR}")

final_r50_640_40ep: build_faster_rcnn_r50, 6 classes, masks=False, best epoch 13
parts_frcnn_640: build_faster_rcnn, 21 classes, masks=False, best epoch 18

detector label sets ready at score >= 0.6


## 6 · Approach 2 — load SmolVLM

In [ ]:
VLM_IDS = {
    "SmolVLM-256M": "HuggingFaceTB/SmolVLM-256M-Instruct",
    "SmolVLM-500M": "HuggingFaceTB/SmolVLM-500M-Instruct",
}


def load_vlm(model_id):
    """Load a SmolVLM checkpoint, tolerating two transformers renames."""
    from transformers import AutoProcessor
    try:
        from transformers import AutoModelForImageTextToText as _Auto
    except ImportError:
        from transformers import AutoModelForVision2Seq as _Auto

    proc = AutoProcessor.from_pretrained(model_id)
    kw = dict(_attn_implementation="eager")   # eager avoids flash-attn, absent on T4
    try:
        model = _Auto.from_pretrained(model_id, dtype=VLM_DTYPE, **kw)
    except TypeError:
        model = _Auto.from_pretrained(model_id, torch_dtype=VLM_DTYPE, **kw)
    model = model.to(DEVICE).eval()
    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"{model_id}: {n:.0f}M parameters")
    return proc, model


@torch.no_grad()
def ask_vlm(proc, model, image_path, question, max_new_tokens=96):
    """One image, one question, greedy decoding so the run is reproducible."""
    img = Image.open(image_path).convert("RGB")
    msg = [{"role": "user", "content": [{"type": "image"},
                                        {"type": "text", "text": question}]}]
    text = proc.apply_chat_template(msg, add_generation_prompt=True, tokenize=False)
    inputs = proc(text=text, images=[img], return_tensors="pt").to(DEVICE)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new = out[0][inputs["input_ids"].shape[1]:]          # strip the prompt
    return proc.decode(new, skip_special_tokens=True).strip()


_proc, _model = load_vlm(VLM_IDS["SmolVLM-500M"])
_t = time.time()
print("\n--- smoke test ---")
print("image :", damage_paths[0].name)
print("truth :", sorted(damage_truth[0]))
print("answer:", ask_vlm(_proc, _model, damage_paths[0],
                         "Describe the damage visible on this vehicle in one sentence."))
print(f"({time.time() - _t:.1f}s for one image)")
del _proc, _model
gc.collect(); torch.cuda.empty_cache()

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.3k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

HuggingFaceTB/SmolVLM-500M-Instruct: 507M parameters

--- smoke test ---
image : Car damages 106.png
truth : ['Broken part', 'Dent', 'Paint chip', 'Scratch']
answer: The vehicle has damage to the front tire.
(1.5s for one image)


## 7 · Test

The prompt hands the model the **closed vocabulary**. It cannot be marked wrong for saying
"scratches" when the class is `Scratch`, because it has been told the exact words to use.

In [9]:
def vocab_prompt(classes, kind):
    return (
        f"This is a photograph of a car. Which of the following {kind} can you see?\n"
        f"Choose only from this list: {', '.join(classes)}.\n"
        f"Answer with the matching items separated by commas, and nothing else. "
        f"If you see none of them, answer: none."
    )


DAMAGE_PROMPT = vocab_prompt(DAMAGE_CLASSES, "kinds of damage")
PARTS_PROMPT  = vocab_prompt(PARTS_CLASSES, "car parts")
print(DAMAGE_PROMPT)


def run_vlm_split(model_name, paths, prompt, tag, save_every=10):
    """Answer `prompt` for every path. Resumes from the checkpoint on Drive."""
    ckpt = Path(VLM_DIR) / f"{tag}__{model_name}.json"
    answers = json.loads(ckpt.read_text()) if ckpt.exists() else {}
    todo = [p for p in paths if p.name not in answers]
    if not todo:
        print(f"{tag}/{model_name}: all {len(paths)} already done")
        return answers

    print(f"\n{tag}/{model_name}: {len(todo)} of {len(paths)} to do")
    proc, model = load_vlm(VLM_IDS[model_name])
    t0 = time.time()
    try:
        for i, p in enumerate(todo, 1):
            answers[p.name] = ask_vlm(proc, model, p, prompt)
            if i % save_every == 0 or i == len(todo):
                ckpt.write_text(json.dumps(answers, indent=1))
                rate = (time.time() - t0) / i
                print(f"  {i}/{len(todo)}  {rate:.2f}s/image  "
                      f"eta {rate * (len(todo) - i) / 60:.1f} min")
    finally:
        ckpt.write_text(json.dumps(answers, indent=1))
        del proc, model
        gc.collect(); torch.cuda.empty_cache()

    secs = time.time() - t0
    json.dump({"seconds": secs, "images": len(todo), "per_image": secs / len(todo)},
              open(Path(VLM_DIR) / f"{tag}__{model_name}__timing.json", "w"), indent=1)
    print(f"  done in {secs / 60:.1f} min ({secs / len(todo):.2f}s/image)")
    return answers


vlm_answers = {}
for _name in ("SmolVLM-500M", "SmolVLM-256M"):
    vlm_answers[(_name, "damage")] = run_vlm_split(_name, damage_paths, DAMAGE_PROMPT, "damage")
    vlm_answers[(_name, "parts")]  = run_vlm_split(_name, parts_paths,  PARTS_PROMPT,  "parts")

print("\n--- five raw answers, damage, SmolVLM-500M ---")
for _i in range(min(5, len(damage_paths))):
    _p = damage_paths[_i]
    print(f"\n{_p.name}\n  truth : {sorted(damage_truth[_i])}"
          f"\n  said  : {vlm_answers[('SmolVLM-500M', 'damage')][_p.name]!r}")

This is a photograph of a car. Which of the following kinds of damage can you see?
Choose only from this list: Missing part, Broken part, Scratch, Other damage, Dent, Paint chip.
Answer with the matching items separated by commas, and nothing else. If you see none of them, answer: none.
damage/SmolVLM-500M: all 81 already done
parts/SmolVLM-500M: all 100 already done
damage/SmolVLM-256M: all 81 already done
parts/SmolVLM-256M: all 100 already done

--- five raw answers, damage, SmolVLM-500M ---

Car damages 106.png
  truth : ['Broken part', 'Dent', 'Paint chip', 'Scratch']
  said  : 'Paint chip.'

Car damages 1063.png
  truth : ['Broken part', 'Dent', 'Missing part', 'Scratch']
  said  : 'Paint chip.'

Car damages 1065.png
  truth : ['Dent', 'Scratch']
  said  : 'Paint chip.'

Car damages 1081.png
  truth : ['Broken part', 'Dent']
  said  : 'Paint chip.'

Car damages 1086.png
  truth : ['Broken part', 'Dent', 'Missing part', 'Scratch']
  said  : 'Broken part.'


In [ ]:
TOKEN_SYNONYMS = {
    # parts
    "windshield": ["windscreen", "front glass", "windshield glass"],
    "hood": ["bonnet"],
    "trunk": ["boot", "tailgate", "trunk lid"],
    "fender": ["wing", "quarter panel", "mudguard"],
    "mirror": ["side mirror", "wing mirror", "rearview mirror", "rear view mirror"],
    "grille": ["grill", "radiator grille", "front grill"],
    "headlight": ["headlamp", "head lamp", "head light"],
    "taillight": ["taillamp", "tail lamp", "tail light", "rear light", "rear lamp"],
    "wheel": ["tyre", "tire", "rim", "alloy", "alloy wheel"],
    "window": ["glass", "window glass"],
    "glass": ["window"],
    "lamp": ["light"],
    "light": ["lamp"],
    "panel": ["body panel"],
    "plate": ["number plate", "registration plate", "licence plate"],
    "sill": ["rocker panel", "side skirt"],
    "roof": ["top"],
    # damage
    "scratch": ["scratches", "scratched", "scrape", "scraped", "scuff", "scuffed",
                "abrasion", "scratching"],
    "dent": ["dents", "dented", "ding", "dinged", "dint", "denting"],
    "chip": ["chipped", "chipping", "chips"],
    "crack": ["cracked", "cracking", "cracks", "fracture", "fractured"],
    "broken": ["break", "shattered", "smashed", "broke"],
    "missing": ["absent", "detached", "torn off", "fallen off"],
    "flaking": ["flaked", "flake", "peeling", "peeled", "peel"],
    "corrosion": ["rust", "rusted", "rusty", "oxidation", "corroded"],
    "damage": ["damaged", "damages"],
}


NEGATORS = {"no", "not", "none", "without", "nor", "never", "neither", "cannot",
            "cant", "isn", "aren", "wasn", "doesn", "don", "didn",
            "free", "absent", "lacks", "lacking",
            # post-positive: "the windshield is intact"
            "intact", "undamaged", "unharmed", "fine", "ok", "okay", "clean",
            "pristine", "unaffected"}


def _norm(s):
    s = unicodedata.normalize("NFKC", s).lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


def _plural(word):
    """'scratch' -> 'scratches', not 'scratchs'. A naive +s silently loses every
    plural answer the model gives, which is most of them."""
    if re.search(r"(s|x|z|ch|sh)$", word):
        return word + "es"
    if re.search(r"[^aeiou]y$", word):
        return word[:-1] + "ies"
    return word + "s"


def _clauses(answer):
    """Split on punctuation and contrastive conjunctions, then normalise each
    piece. Normalising first would destroy the boundaries."""
    raw = unicodedata.normalize("NFKC", answer).lower()
    pieces = re.split(r"[.,;:!?\n()]+|\bbut\b|\bhowever\b|\balthough\b|\bwhereas\b", raw)
    out = [re.sub(r"[^a-z0-9]+", " ", p).strip() for p in pieces]
    return [p for p in out if p]


def _mentions(clauses, phrase):
    """True if `phrase` appears as whole words in a clause that is not negated."""
    pat = re.compile(rf"(?<![a-z0-9]){re.escape(phrase)}(?![a-z0-9])")
    for c in clauses:
        if pat.search(c) and not any(t in NEGATORS for t in c.split()):
            return True
    return False


def build_aliases(classes, lenient):
    """class name -> phrases that count as naming it. Family words are handled
    separately in parse_answer, because a bare 'bumper' is ambiguous between
    classes and has to be resolved per image."""
    aliases = {}
    for c in classes:
        toks = _norm(c).split()
        forms = {" ".join(toks), " ".join(toks[:-1] + [_plural(toks[-1])])}
        if lenient:
            for i, t in enumerate(toks):
                for syn in TOKEN_SYNONYMS.get(t, []):
                    forms.add(" ".join(toks[:i] + [syn] + toks[i + 1:]))
            if len(toks) == 1:
                forms.update(TOKEN_SYNONYMS.get(toks[0], []))
        aliases[c] = sorted(forms, key=len, reverse=True)
    return aliases


def build_families(classes):
    """Data-driven, no hand-written list: classes sharing a last token form a
    family, and the shared token is the ambiguous bare word. 'Front-bumper' and
    'Back-bumper' -> 'bumper'. Single-member families are not ambiguous."""
    groups = {}
    for c in classes:
        toks = _norm(c).split()
        if len(toks) > 1:
            groups.setdefault(toks[-1], set()).add(c)
    return {w: m for w, m in groups.items() if len(m) > 1}


def parse_answer(answer, classes, aliases, families, truth=frozenset(), lenient=False):
    """Text -> (set of class names, the subset credited only via a family word)."""
    text = _norm(answer)
    if not text or text in ("none", "no", "nothing", "none of them"):
        return set(), set()
    clauses = _clauses(answer)

    found = {c for c in classes if any(_mentions(clauses, a) for a in aliases[c])}

    family_credit = set()
    if lenient:
        for word, members in families.items():
            if found & members:
                continue                       # already named a specific one
            if _mentions(clauses, word):
                # Generous: credit whichever family member is actually present.
                family_credit |= members & set(truth)
    return found | family_credit, family_credit


def set_scores(truth_sets, pred_sets):
    tp = sum(len(t & p) for t, p in zip(truth_sets, pred_sets))
    n_t = sum(len(t) for t in truth_sets)
    n_p = sum(len(p) for p in pred_sets)
    rec = tp / n_t if n_t else 0.0
    prec = tp / n_p if n_p else 0.0
    return {
        "recall": rec,
        "precision": prec,
        "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
        "any correct": sum(bool(t & p) for t, p in zip(truth_sets, pred_sets)) / len(truth_sets),
        "exact set": sum(t == p for t, p in zip(truth_sets, pred_sets)) / len(truth_sets),
        "said nothing": sum(not p for p in pred_sets) / len(truth_sets),
        "labels/image": n_p / len(truth_sets),
        "truth/image": n_t / len(truth_sets),
    }


def show_scores(table, title):
    cols = ["recall", "precision", "f1", "any correct", "exact set",
            "said nothing", "labels/image", "truth/image"]
    width = max(len(k) for k in table) + 2
    print(f"\n{title}")
    print(" " * width + "".join(f"{c:>14s}" for c in cols))
    for name, s in table.items():
        print(f"{name:<{width}}" + "".join(f"{s[c]:>14.2f}" for c in cols))


def compare(kind, classes, paths, truth, det_sets, run_name):
    al_s, al_l = build_aliases(classes, False), build_aliases(classes, True)
    fams = build_families(classes)
    table, echoed, family_hits = {}, {}, {}

    table[f"detector {run_name}"] = set_scores(truth, det_sets)

    for model in ("SmolVLM-500M", "SmolVLM-256M"):
        answers = vlm_answers[(model, kind)]
        for level, al in (("strict", al_s), ("lenient", al_l)):
            sets, credits = [], 0
            for p, t in zip(paths, truth):
                s, fc = parse_answer(answers[p.name], classes, al, fams, t,
                                     lenient=(level == "lenient"))
                sets.append(s); credits += len(fc)
            table[f"{model} ({level})"] = set_scores(truth, sets)
            if level == "lenient":
                family_hits[model] = credits
        # how often did it simply read the vocabulary back?
        echoed[model] = sum(
            len(parse_answer(answers[p.name], classes, al_s, fams)[0]) >= max(3, len(classes) // 2)
            for p in paths) / len(paths)

    show_scores(table, f"{kind.upper()} — label sets on {len(truth)} held-out images "
                       f"(detector at score {SCORE_THR}, boxes discarded)")
    print(f"\n  ambiguous families in this vocabulary: "
          f"{ {w: sorted(m) for w, m in fams.items()} }")
    for m in echoed:
        print(f"  {m}: echoed half the vocabulary or more on {echoed[m]:.0%} of images; "
              f"{family_hits[m]} labels credited only via a bare family word")
    return table


damage_cmp = compare("damage", DAMAGE_CLASSES, damage_paths, damage_truth,
                     damage_det, DAMAGE_RUN)
parts_cmp  = compare("parts", PARTS_CLASSES, parts_paths, parts_truth,
                     parts_det, PARTS_RUN)


DAMAGE — label sets on 81 held-out images (detector at score 0.6, boxes discarded)
                                     recall     precision            f1   any correct     exact set  said nothing  labels/image   truth/image
detector final_r50_640_40ep            0.60          0.81          0.69          0.96          0.11          0.01          2.52          3.41
SmolVLM-500M (strict)                  0.12          0.40          0.18          0.40          0.00          0.00          1.00          3.41
SmolVLM-500M (lenient)                 0.12          0.40          0.18          0.40          0.00          0.00          1.00          3.41
SmolVLM-256M (strict)                  0.11          0.46          0.18          0.38          0.00          0.16          0.84          3.41
SmolVLM-256M (lenient)                 0.11          0.46          0.18          0.38          0.00          0.16          0.84          3.41

  ambiguous families in this vocabulary: {'part': ['Broken part

## 8 · Does it agree with whatever it is asked?

This is not a detection question, it is the question that decides claims. The
pipeline reads a **cause** and matches it against the insured perils; fire, flood
and theft are separate grants in Section I with different exclusions.

The control question is the one whose answer should be *yes*. Without it, a model
that says yes to everything is indistinguishable from one that is right.

In [ ]:
assert not any(re.search(r"fire|flood|burn|submerg|water", c, re.I)
               for c in DAMAGE_CLASSES), \
    f"a damage class may cover fire or flood: {DAMAGE_CLASSES}"

_counts = Counter(c for t in damage_truth for c in t)
print("held-out images containing each damage class:")
for _c, _n in _counts.most_common():
    print(f"  {_c:<20}{_n:>5}")
print("\nThese are appearance classes on collision/cosmetic photos. Spot-check a few")
print("images before trusting the fire/flood baseline.\n")

PROBES = {
    "fire (expect no)": "Was this vehicle damaged by fire? Answer yes or no.",
    "flood (expect no)": "Was this vehicle damaged by a flood? Answer yes or no.",
    "theft (expect no)": "Was this vehicle broken into and stolen from? Answer yes or no.",
    "collision (expect yes)": "Is there any visible damage to this vehicle? Answer yes or no.",
}


def read_yes_no(answer):
    a = _norm(answer)
    if re.match(r"^(yes|yeah|yep|correct|true)\b", a):
        return True
    if re.match(r"^(no|nope|false|none)\b", a):
        return False
    if re.search(r"(?<![a-z])yes(?![a-z])", a):
        return True
    if re.search(r"(?<![a-z])no(?![a-z])", a):
        return False
    return None             


PROBE_MODEL = "SmolVLM-500M"
probe_ckpt = Path(VLM_DIR) / f"probe__{PROBE_MODEL}.json"
probe_raw = json.loads(probe_ckpt.read_text()) if probe_ckpt.exists() else {}

_todo = [(k, p.name) for k in PROBES for p in damage_paths
         if f"{k}|{p.name}" not in probe_raw]
if _todo:
    _by_name = {p.name: p for p in damage_paths}
    _proc, _model = load_vlm(VLM_IDS[PROBE_MODEL])
    _t = time.time()
    try:
        for _i, (_k, _n) in enumerate(_todo, 1):
            probe_raw[f"{_k}|{_n}"] = ask_vlm(_proc, _model, _by_name[_n], PROBES[_k],
                                              max_new_tokens=16)
            if _i % 20 == 0 or _i == len(_todo):
                probe_ckpt.write_text(json.dumps(probe_raw, indent=1))
                print(f"  {_i}/{len(_todo)}  eta "
                      f"{(time.time() - _t) / _i * (len(_todo) - _i) / 60:.1f} min")
    finally:
        probe_ckpt.write_text(json.dumps(probe_raw, indent=1))
        del _proc, _model
        gc.collect(); torch.cuda.empty_cache()
else:
    print(f"probe already complete ({len(probe_raw)} answers)")

probe_rates = {}
print(f"\n{PROBE_MODEL} on {len(damage_paths)} held-out collision photos\n")
print(f"{'question':<26}{'said yes':>10}{'said no':>10}{'no answer':>12}")
for _k in PROBES:
    _v = [read_yes_no(probe_raw[f"{_k}|{p.name}"]) for p in damage_paths]
    _n = len(_v)
    probe_rates[_k] = {"yes": _v.count(True) / _n, "no": _v.count(False) / _n,
                       "unparsed": _v.count(None) / _n}
    print(f"{_k:<26}{probe_rates[_k]['yes']:>10.0%}{probe_rates[_k]['no']:>10.0%}"
          f"{probe_rates[_k]['unparsed']:>12.0%}")

_false = max(probe_rates[k]["yes"] for k in PROBES if "expect no" in k)
_true = probe_rates["collision (expect yes)"]["yes"]
print(f"\nAgrees with a false premise on up to {_false:.0%} of photos, and with the "
      f"true one on {_true:.0%}.")
print("A gap near zero means the yes carries no information about the photo, and the"
      "\ncause field must come from the policy text and the claim form, never from here.")
json.dump(probe_rates, open(f"{VLM_DIR}/probe_rates.json", "w"), indent=1)

held-out images containing each damage class:
  Broken part            71
  Dent                   67
  Scratch                47
  Missing part           41
  Paint chip             29
  Other damage           21

These are appearance classes on collision/cosmetic photos. Spot-check a few
images before trusting the fire/flood baseline.

probe already complete (324 answers)

SmolVLM-500M on 81 held-out collision photos

question                    said yes   said no   no answer
fire (expect no)                  6%       94%          0%
flood (expect no)                 4%       96%          0%
theft (expect no)                 4%       96%          0%
collision (expect yes)           70%       30%          0%

Agrees with a false premise on up to 6% of photos, and with the true one on 70%.
A gap near zero means the yes carries no information about the photo, and the
cause field must come from the policy text and the claim form, never from here.


## 9 · The comparison, and what it cannot say

In [ ]:
_timing = {}
for _m in ("SmolVLM-500M", "SmolVLM-256M"):
    for _k in ("damage", "parts"):
        _f = Path(VLM_DIR) / f"{_k}__{_m}__timing.json"
        if _f.exists():
            _timing[f"{_m}/{_k}"] = json.loads(_f.read_text())["per_image"]

summary = {
    "damage": damage_cmp,
    "parts": parts_cmp,
    "probe": probe_rates,
    "seconds_per_image": _timing,
    "runs": {"damage": DAMAGE_RUN, "parts": PARTS_RUN},
    "task": ("label sets per image, boxes discarded; detectors at score "
             f"{SCORE_THR}; same held-out splits as the detector evaluation"),
}
json.dump(summary, open(f"{REPORT_DIR}/vlm_vs_detector.json", "w"), indent=1)

print("=" * 78)
print("TWO APPROACHES, SAME TASK: which labels are on this photo")
print("=" * 78)
for _kind, _cmp in (("DAMAGE", damage_cmp), ("PARTS", parts_cmp)):
    print(f"\n{_kind}")
    print(f"  {'approach':<28}{'recall':>9}{'precision':>11}{'f1':>8}{'labels/img':>12}")
    for _name, _s in _cmp.items():
        print(f"  {_name:<28}{_s['recall']:>9.2f}{_s['precision']:>11.2f}"
              f"{_s['f1']:>8.2f}{_s['labels/image']:>12.1f}")

print("\ncost per image")
for _k, _v in _timing.items():
    print(f"  {_k:<28}{_v:>6.2f}s")

_pir = Path(REPORT_DIR) / "per_image_rates.json"
if _pir.exists():
    _d = json.loads(_pir.read_text())
    _row = next((r for r in _d["damage"] if abs(r["threshold"] - SCORE_THR) < 1e-9), None)
    if _row:
        print(f"\nFor contrast, the detector on the task the VLM cannot attempt at all")
        print(f"(damage, localise AND name, score {SCORE_THR}): "
              f"located {_row['located']:.2f}, right type {_row['right type']:.2f}")
else:
    print("\n(per_image_rates.json not on Drive yet — run the main notebook's"
          " evaluation cells for the localisation numbers)")

print("""
WHAT THESE NUMBERS DO NOT CAPTURE, and the write-up must say so:

1. The VLM returns no boxes. This table is a label-set task, which is strictly
   easier than detection, and the detectors were handicapped into it by throwing
   their boxes away. On the actual task - localise the damage and localise the
   part - the VLM scores nothing, because it produces nothing to score.

2. linking.py cannot run on a VLM. The pipeline attaches damage to parts by box
   containment. Without boxes there is no "scratch on the FRONT BUMPER" and no
   "dent on the DOOR", only an unordered bag of words, so the depreciation table
   cannot be applied per part and the claim cannot be itemised.

3. Every allowance made in the VLM's favour turned out to be dead weight:
   strict and lenient score identically and one single label across both models
   was credited via a bare family word. It never names enough to reach them.
   Recall is also capped by construction - see cell 24.

4. The probe above is the finding that settles the cause field.
""")
print(f"saved -> {REPORT_DIR}/vlm_vs_detector.json")

TWO APPROACHES, SAME TASK: which labels are on this photo

DAMAGE
  approach                       recall  precision      f1  labels/img
  detector final_r50_640_40ep      0.60       0.81    0.69         2.5
  SmolVLM-500M (strict)            0.12       0.40    0.18         1.0
  SmolVLM-500M (lenient)           0.12       0.40    0.18         1.0
  SmolVLM-256M (strict)            0.11       0.46    0.18         0.8
  SmolVLM-256M (lenient)           0.11       0.46    0.18         0.8

PARTS
  approach                       recall  precision      f1  labels/img
  detector parts_frcnn_640         0.85       0.97    0.90        12.9
  SmolVLM-500M (strict)            0.05       0.69    0.09         1.0
  SmolVLM-500M (lenient)           0.05       0.69    0.09         1.0
  SmolVLM-256M (strict)            0.04       0.68    0.08         1.0
  SmolVLM-256M (lenient)           0.05       0.68    0.09         1.0

cost per image
  SmolVLM-500M/damage           0.69s
  SmolVLM-500M/parts 

## 10 · Is the VLM looking at the photo?

Cell 19 reported `labels/image` = **1.00** for both SmolVLM sizes, against a
truth average of 3.41 (damage) and 14.73 (parts). That caps micro-recall before
the model gets anything right, so recall is the wrong place to look.

Precision is the fair read and the test below is whether that precision beats
**ignoring the image entirely**. For every class, "always answer this one class"
is scored as a strategy. If the best constant matches SmolVLM, then on this task
SmolVLM carries no information from the pixels.

In [ ]:
def load_answers(model_name, tag):
    return json.loads((Path(VLM_DIR) / f"{tag}__{model_name}.json").read_text())


def recall_ceiling(truth, labels_per_image=1.0):
    """Highest micro-recall reachable while emitting this many labels per image,
    even if every one of them is correct."""
    n_t = sum(len(t) for t in truth)
    return min(1.0, labels_per_image * len(truth) / n_t) if n_t else 0.0


def constant_baselines(classes, truth):
    """Score every 'always answer exactly this one class' strategy."""
    rows = {}
    for c in classes:
        rows[c] = set_scores(truth, [{c}] * len(truth))
    return dict(sorted(rows.items(), key=lambda kv: -kv[1]["f1"]))


def diagnose(kind, classes, paths, truth):
    print("=" * 78)
    print(f"{kind.upper()} — is the answer coming from the photo?")
    print("=" * 78)

    print(f"\ntruth averages {sum(len(t) for t in truth) / len(truth):.2f} labels/image")
    for n in (1.0, 2.0, 3.0):
        print(f"  emitting {n:.0f} label(s)/image caps micro-recall at "
              f"{recall_ceiling(truth, n):.2f}")

    al = build_aliases(classes, False)
    fams = build_families(classes)

    for model in ("SmolVLM-500M", "SmolVLM-256M"):
        answers = load_answers(model, kind)
        raw = [answers[p.name] for p in paths]
        norm = [_norm(a) for a in raw]
        tally = Counter(norm)
        parsed = [parse_answer(a, classes, al, fams)[0] for a in raw]
        s = set_scores(truth, parsed)

        print(f"\n{model}")
        print(f"  distinct answers        {len(tally)} across {len(raw)} images")
        for text, n in tally.most_common(5):
            print(f"    {n:>4} ({n / len(raw):>4.0%})  {text[:58]!r}")
        print(f"  mode share              {tally.most_common(1)[0][1] / len(raw):.0%}")
        print(f"  its precision           {s['precision']:.2f}")

        # The decisive comparison.
        best = constant_baselines(classes, truth)
        top_name, top = next(iter(best.items()))
        print(f"  best constant baseline  always say {top_name!r}: "
              f"precision {top['precision']:.2f}, recall {top['recall']:.2f}, "
              f"f1 {top['f1']:.2f}")
        print(f"  {model} f1            {s['f1']:.2f}")
        verdict = ("NO BETTER than ignoring the image"
                   if s["f1"] <= top["f1"] + 0.02 else
                   f"beats the best constant by {s['f1'] - top['f1']:+.2f} f1")
        print(f"  verdict                 {verdict}")

    print("\n  every constant baseline, best f1 first:")
    print(f"    {'always answer':<20}{'precision':>11}{'recall':>9}{'f1':>7}")
    for c, s in constant_baselines(classes, truth).items():
        print(f"    {c:<20}{s['precision']:>11.2f}{s['recall']:>9.2f}{s['f1']:>7.2f}")


diagnose("damage", DAMAGE_CLASSES, damage_paths, damage_truth)
print()
diagnose("parts", PARTS_CLASSES, parts_paths, parts_truth)

DAMAGE — is the answer coming from the photo?

truth averages 3.41 labels/image
  emitting 1 label(s)/image caps micro-recall at 0.29
  emitting 2 label(s)/image caps micro-recall at 0.59
  emitting 3 label(s)/image caps micro-recall at 0.88

SmolVLM-500M
  distinct answers        6 across 81 images
      47 ( 58%)  'paint chip'
      22 ( 27%)  'other damage'
       9 ( 11%)  'scratch'
       1 (  1%)  'broken part'
       1 (  1%)  'dent'
  mode share              58%
  its precision           0.40
  best constant baseline  always say 'Broken part': precision 0.88, recall 0.26, f1 0.40
  SmolVLM-500M f1            0.18
  verdict                 NO BETTER than ignoring the image

SmolVLM-256M
  distinct answers        5 across 81 images
      53 ( 65%)  'missing part'
      13 ( 16%)  'none'
       8 ( 10%)  'other damage'
       5 (  6%)  'scratch'
       2 (  2%)  'paint chip'
  mode share              65%
  its precision           0.46
  best constant baseline  always say 'Broken p

## 11 · Did it get a fair shot?

In [ ]:
PROBE_PARTS_TOO = False 

PROMPT_V2 = {
    "damage": ("This is a photograph of a car. Several kinds of damage are usually "
               "visible at once. List EVERY one you can see from this list: "
               f"{', '.join(DAMAGE_CLASSES)}. "
               "Separate them with commas. Most photos have three or more."),
    "parts": ("This is a photograph of a car. Many parts are visible at once. "
              "List EVERY part you can see from this list: "
              f"{', '.join(PARTS_CLASSES)}. "
              "Separate them with commas. Most photos show ten or more."),
}

V2_MODEL = "SmolVLM-500M"


def run_prompt_v2(kind, paths, save_every=20):
    ckpt = Path(VLM_DIR) / f"{kind}_v2__{V2_MODEL}.json"
    answers = json.loads(ckpt.read_text()) if ckpt.exists() else {}
    todo = [p for p in paths if p.name not in answers]
    if todo:
        proc, model = load_vlm(VLM_IDS[V2_MODEL])
        t0 = time.time()
        try:
            for i, p in enumerate(todo, 1):
                answers[p.name] = ask_vlm(proc, model, p, PROMPT_V2[kind],
                                          max_new_tokens=160)
                if i % save_every == 0 or i == len(todo):
                    ckpt.write_text(json.dumps(answers, indent=1))
                    print(f"  v2 {kind}: {i}/{len(todo)}  "
                          f"{(time.time() - t0) / i:.2f}s/image")
        finally:
            ckpt.write_text(json.dumps(answers, indent=1))
            del proc, model
            gc.collect(); torch.cuda.empty_cache()
    else:
        print(f"  v2 {kind}: already complete")
    return answers


def run_prompt_v3(kind, classes, paths, save_every=50):
    """One yes/no question per class per image."""
    ckpt = Path(VLM_DIR) / f"{kind}_v3__{V2_MODEL}.json"
    answers = json.loads(ckpt.read_text()) if ckpt.exists() else {}
    todo = [(c, p.name) for c in classes for p in paths
            if f"{c}|{p.name}" not in answers]
    if todo:
        by_name = {p.name: p for p in paths}
        proc, model = load_vlm(VLM_IDS[V2_MODEL])
        t0 = time.time()
        try:
            for i, (c, n) in enumerate(todo, 1):
                q = (f"This is a photograph of a car. Looking only at this photo, "
                     f"is there a {c.replace('-', ' ').lower()} visible? "
                     f"Answer yes or no.")
                answers[f"{c}|{n}"] = ask_vlm(proc, model, by_name[n], q,
                                              max_new_tokens=16)
                if i % save_every == 0 or i == len(todo):
                    ckpt.write_text(json.dumps(answers, indent=1))
                    print(f"  v3 {kind}: {i}/{len(todo)}  eta "
                          f"{(time.time() - t0) / i * (len(todo) - i) / 60:.1f} min")
        finally:
            ckpt.write_text(json.dumps(answers, indent=1))
            del proc, model
            gc.collect(); torch.cuda.empty_cache()
    else:
        print(f"  v3 {kind}: already complete ({len(answers)} answers)")
    return answers


def v3_label_sets(answers, classes, paths):
    """A 'yes' per class becomes a label. Unparseable answers count as no."""
    sets, unparsed = [], 0
    for p in paths:
        s = set()
        for c in classes:
            v = read_yes_no(answers[f"{c}|{p.name}"])
            if v is None:
                unparsed += 1
            elif v:
                s.add(c)
        sets.append(s)
    return sets, unparsed


def fairness(kind, classes, paths, truth, det_sets, run_name):
    al = build_aliases(classes, True)
    fams = build_families(classes)
    table = {f"detector {run_name}": set_scores(truth, det_sets)}

    v1 = [parse_answer(vlm_answers[(V2_MODEL, kind)][p.name], classes, al, fams,
                       t, lenient=True)[0] for p, t in zip(paths, truth)]
    table["v1 closed vocabulary"] = set_scores(truth, v1)

    a2 = run_prompt_v2(kind, paths)
    v2 = [parse_answer(a2[p.name], classes, al, fams, t, lenient=True)[0]
          for p, t in zip(paths, truth)]
    table["v2 'list every one'"] = set_scores(truth, v2)

    a3 = run_prompt_v3(kind, classes, paths)
    v3, unparsed = v3_label_sets(a3, classes, paths)
    table["v3 one yes/no per class"] = set_scores(truth, v3)

    show_scores(table, f"{kind.upper()} — {V2_MODEL} under three promptings "
                       f"({len(paths)} images)")
    print(f"\n  v3 asked {len(classes)} x {len(paths)} = {len(classes) * len(paths)} "
          f"questions; {unparsed} answers were not yes or no (counted as no)")
    print(f"  v3 removes the recall ceiling entirely: it may emit up to "
          f"{len(classes)} labels per image")
    return table


damage_fair = fairness("damage", DAMAGE_CLASSES, damage_paths, damage_truth,
                       damage_det, DAMAGE_RUN)
if PROBE_PARTS_TOO:
    parts_fair = fairness("parts", PARTS_CLASSES, parts_paths, parts_truth,
                          parts_det, PARTS_RUN)

json.dump({"damage": damage_fair,
           **({"parts": parts_fair} if PROBE_PARTS_TOO else {})},
          open(f"{REPORT_DIR}/vlm_prompt_fairness.json", "w"), indent=1)
print(f"\nsaved -> {REPORT_DIR}/vlm_prompt_fairness.json")

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

HuggingFaceTB/SmolVLM-500M-Instruct: 507M parameters
  v2 damage: 20/81  1.04s/image
  v2 damage: 40/81  1.06s/image
  v2 damage: 60/81  1.09s/image
  v2 damage: 80/81  1.11s/image
  v2 damage: 81/81  1.12s/image


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

HuggingFaceTB/SmolVLM-500M-Instruct: 507M parameters
  v3 damage: 50/486  eta 4.8 min
  v3 damage: 100/486  eta 4.1 min
  v3 damage: 150/486  eta 3.6 min
  v3 damage: 200/486  eta 3.1 min
  v3 damage: 250/486  eta 2.5 min
  v3 damage: 300/486  eta 2.0 min
  v3 damage: 350/486  eta 1.5 min
  v3 damage: 400/486  eta 0.9 min
  v3 damage: 450/486  eta 0.4 min
  v3 damage: 486/486  eta 0.0 min

DAMAGE — SmolVLM-500M under three promptings (81 images)
                                     recall     precision            f1   any correct     exact set  said nothing  labels/image   truth/image
detector final_r50_640_40ep            0.60          0.81          0.69          0.96          0.11          0.01          2.52          3.41
v1 closed vocabulary                   0.12          0.40          0.18          0.40          0.00          0.00          1.00          3.41
v2 'list every one'                    0.80          0.59          0.68          0.85          0.05          0.01          4